# 04 · Evaluate: GAN vs diffusion on an unseen Kerala event

Four ways of producing the after image of the **test event** (never seen in training):

| Method | What it is |
|---|---|
| **pix2pix** (GAN) | trained on before/after pairs with slope, rain, soil, severity |
| **Conditional diffusion** | same data and inputs, Palette-style DDPM sampled with DDIM |
| **Identity** baseline | "nothing changes": after = before. Both models must beat it. |
| **Stable Diffusion img2img** | large pretrained model, no task training, text prompt only |

Metrics: SSIM, PSNR, FID (on RGB), generation time, and **severity consistency** (does the
measured damage follow the requested severity?). Plus a human real-vs-generated survey.
Every number in the report comes from `outputs/metrics.json`. Runtime: **GPU (T4)**.

In [ ]:
from google.colab import drive

REPO_URL = "https://github.com/muhammedshahidsds25-collab/kerala-hazard-vision-ai.git"
REPO_DIR = "/content/kerala-hazard-vision-ai"

drive.mount("/content/drive")
!git clone -q {REPO_URL} {REPO_DIR} || git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
# Regular (not editable) install: an editable src-layout install is not importable
# until the runtime restarts.
!pip install -q ".[sd]"

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from kerala_hazard_vision_ai.config import load_config
from kerala_hazard_vision_ai.data.dataset import (
    ChannelStats,
    TileDataset,
    split_by_event,
    to_reflectance,
    unnormalise_before,
)
from kerala_hazard_vision_ai.diffusion_baseline import load_pipeline, render_prompts, sd_img2img
from kerala_hazard_vision_ai.evaluate import (
    SEVERITY_CHANNEL,
    diffusion_predict,
    evaluate_identity,
    evaluate_model,
    evaluate_rgb,
    gan_predict,
    plot_comparison,
    plot_severity_sweep,
    rgb_uint8,
    save_metrics,
)
from kerala_hazard_vision_ai.human_eval import build_survey, save_rgb_pngs, score_survey
from kerala_hazard_vision_ai.models import UNetGenerator
from kerala_hazard_vision_ai.train_diffusion import build_diffusion

cfg = load_config("configs/default.yaml")
DRIVE_ROOT = Path("/content/drive/MyDrive") / cfg.data.drive_folder
OUT_DIR = DRIVE_ROOT / "outputs"
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_weights(model: torch.nn.Module, path: Path) -> torch.nn.Module:
    model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    return model


# Both models were trained on the same split, so their stats.json files are identical.
stats = ChannelStats.load(DRIVE_ROOT / "checkpoints" / "stats.json")
split = split_by_event(
    sorted((DRIVE_ROOT / "tiles").glob("*.npz")),
    {cfg.data.test_event},
    cfg.train.val_frac,
    cfg.data.tile_size,
)
test_ds = TileDataset(split["test"], stats, augment=False)
test_event = next(e for e in cfg.events if e.name == cfg.data.test_event)

generator = UNetGenerator(cfg.model.in_channels, cfg.model.out_channels, cfg.model.ngf,
                          cfg.model.num_downs)
generator = load_weights(generator, DRIVE_ROOT / "checkpoints" / "best_g.pt")
diffusion = load_weights(build_diffusion(cfg), DRIVE_ROOT / "checkpoints_diffusion" / "best_ema.pt")
predictors = {
    "pix2pix": gan_predict(generator, device),
    "diffusion": diffusion_predict(diffusion, device, cfg.diffusion.sample_steps, cfg.train.seed),
}
print(f"Test event: {test_event.name} ({test_event.hazard}), {len(test_ds)} tiles")

## 1. pix2pix vs diffusion vs identity
The diffusion model runs 50 denoising steps per image, so this cell takes longer.

In [ ]:
results = {"test_event": test_event.name}
sweeps = {}
for name, predict in predictors.items():
    results[name] = evaluate_model(predict, test_ds, cfg)
    sweeps[name] = results[name].pop("severity_sweep")  # raw points: plotted, not tabled
results["identity"] = evaluate_identity(test_ds, cfg)
pd.DataFrame({k: results[k] for k in ("pix2pix", "diffusion", "identity")}).T

## 2. Figures

In [ ]:
severe_first = test_ds.severities.argsort()[::-1][:4].tolist()
x, y = (torch.stack(t) for t in zip(*(test_ds[i] for i in severe_first), strict=True))


def to_rgb(scaled: torch.Tensor) -> torch.Tensor:
    return rgb_uint8(to_reflectance(scaled, stats), cfg.eval.rgb_max)


before_rgb = rgb_uint8(unnormalise_before(x, stats), cfg.eval.rgb_max)
model_rgb = {name: to_rgb(predict(x)) for name, predict in predictors.items()}
plot_comparison({"Before": before_rgb, "Real after": to_rgb(y), "pix2pix": model_rgb["pix2pix"],
                 "Diffusion": model_rgb["diffusion"]}, FIG_DIR / "comparison.png")

In [ ]:
for name, predict in predictors.items():
    frames = []
    for sev in cfg.demo.sweep:
        swept = x[:1].clone()
        swept[:, SEVERITY_CHANNEL] = sev
        frames.append(to_rgb(predict(swept))[0])
    display(plot_severity_sweep(before_rgb[0], frames, cfg.demo.sweep,
                                FIG_DIR / f"sweep_{name}.png"))

**Diversity:** diffusion can show several plausible outcomes for the same area and severity
(different random seeds). pix2pix always gives the same answer.

In [ ]:
steps = cfg.diffusion.sample_steps
samples = {f"Seed {s}": to_rgb(diffusion_predict(diffusion, device, steps, s)(x)) for s in range(4)}
plot_comparison({"Before": before_rgb, **samples}, FIG_DIR / "diffusion_diversity.png")

In [ ]:
sweep_df = pd.concat(
    [pd.DataFrame(points).assign(model=name) for name, points in sweeps.items()]
)
fig, axes = plt.subplots(1, len(sweeps), figsize=(5 * len(sweeps), 4), sharey=True)
for ax, (name, group) in zip(axes, sweep_df.groupby("model"), strict=True):
    group.boxplot(column="measured", by="requested", ax=ax)
    ax.set_title(name)
    ax.set_xlabel("requested severity")
axes[0].set_ylabel("measured severity of generated image")
fig.suptitle("Severity control")
fig.tight_layout()
fig.savefig(FIG_DIR / "severity_consistency.png", dpi=150)

## 3. Stable Diffusion img2img (zero-shot, prompt engineering)
Same before images, 4 prompt designs from `configs/default.yaml`. Takes ~20–30 min on a T4.

In [ ]:
pipe = load_pipeline(cfg.eval, device)
all_x, all_y = (torch.stack(t) for t in zip(*test_ds, strict=True))
test_before_rgb = rgb_uint8(unnormalise_before(all_x, stats), cfg.eval.rgb_max)
test_real_rgb = to_rgb(all_y)

results["stable_diffusion"] = {}
sd_examples = {}
for prompt in render_prompts(cfg.eval.prompts, test_event.hazard):
    sd_rgb = sd_img2img(pipe, test_before_rgb, prompt, cfg.eval, seed=cfg.train.seed)
    results["stable_diffusion"][prompt] = evaluate_rgb(sd_rgb, test_real_rgb)
    sd_examples[prompt] = sd_rgb[severe_first]
del pipe
torch.cuda.empty_cache()
pd.DataFrame(results["stable_diffusion"]).T

In [ ]:
# Chosen on the test set: this favours the baseline, which only makes our models' win harder.
best_prompt = max(results["stable_diffusion"], key=lambda p: results["stable_diffusion"][p]["ssim"])
results["stable_diffusion_best_prompt"] = best_prompt
plot_comparison({"Before": before_rgb, "Real after": to_rgb(y),
                 "Stable Diffusion": sd_examples[best_prompt]},
                FIG_DIR / "comparison_stable_diffusion.png")
best_prompt

## 4. Results table → `outputs/metrics.json`

In [ ]:
save_metrics(results, OUT_DIR / "metrics.json")
table = {
    "pix2pix (GAN)": results["pix2pix"],
    "Conditional diffusion": results["diffusion"],
    "Identity baseline": results["identity"],
    "Stable Diffusion (best prompt)": results["stable_diffusion"][best_prompt],
}
pd.DataFrame(table).T[["n_tiles", "ssim", "psnr", "fid", "sec_per_tile"]].round(3)

## 5. Human evaluation survey
Builds 21 images in `outputs/human_eval/`: 7 real, 7 pix2pix, 7 diffusion, shuffled, each from
a different tile. Share `q01.png`…`q21.png` with 5–10 classmates (e.g. a Google Form with one
"Real / AI-generated" question per image). Do **not** share `key.csv`.

In [ ]:
SURVEY_DIR = OUT_DIR / "human_eval"
pools = {"real": test_real_rgb}
for name, predict in predictors.items():
    pools[name] = torch.cat([to_rgb(predict(all_x[i : i + 16])) for i in range(0, len(all_x), 16)])
pngs = {name: save_rgb_pngs(rgb, SURVEY_DIR / "pool", name) for name, rgb in pools.items()}
real_pngs = pngs.pop("real")
build_survey(real_pngs, pngs, n_per_source=7, out_dir=SURVEY_DIR, seed=cfg.train.seed)

After collecting answers, save them as `outputs/human_eval/answers.csv` with columns
`rater,item,answer` (answer = `real` or `generated`), then run:

In [ ]:
results["human_eval"] = score_survey(SURVEY_DIR / "answers.csv", SURVEY_DIR / "key.csv")
save_metrics(results, OUT_DIR / "metrics.json")
results["human_eval"]

Copy `outputs/metrics.json`, `outputs/figures/` and `outputs/human_eval/key.csv` +
`answers.csv` from Drive into the repo's `outputs/` folder, save this executed notebook into
`notebooks/`, then commit.